### IMPORT LIBRARIES ###

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

nltk.download('stopwords')
print("✅ Đã import xong bộ thư viện mới (No Newspaper)!")

✅ Đã import xong bộ thư viện mới (No Newspaper)!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


### Load and EDA Data ###

In [25]:
import os
import pandas as pd

datasets = []
dataset_folder = '../Datasets'

for i in range(16, 27):
    file_name = f'Dataset{i}.csv'
    file_path = os.path.join(dataset_folder, file_name)
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        # Chuyển tất cả tên cột về chữ thường (lowercase)
        df.columns = [col.lower() for col in df.columns]
        datasets.append(df)
        print(f"✅ Loaded & Lowercased columns: {file_name}")

# Gộp tất cả lại
df_news = pd.concat(datasets, ignore_index=True)
print(f"\n📊 Tổng cộng: {df_news.shape[0]} dòng.")

✅ Loaded & Lowercased columns: Dataset16.csv
✅ Loaded & Lowercased columns: Dataset17.csv
✅ Loaded & Lowercased columns: Dataset18.csv
✅ Loaded & Lowercased columns: Dataset19.csv
✅ Loaded & Lowercased columns: Dataset20.csv
✅ Loaded & Lowercased columns: Dataset21.csv
✅ Loaded & Lowercased columns: Dataset22.csv
✅ Loaded & Lowercased columns: Dataset23.csv
✅ Loaded & Lowercased columns: Dataset24.csv
✅ Loaded & Lowercased columns: Dataset25.csv
✅ Loaded & Lowercased columns: Dataset26.csv

📊 Tổng cộng: 522419 dòng.


In [27]:
# Chú thích: Hàm này giúp biến các ký hiệu viết tắt (e, b, t, m) thành chữ dễ đọc.
# 'e' -> Entertainment, 'b' -> Business, 't' -> Technology, 'm' -> Health

mapping = {'e': 'Entertainment', 'b': 'Business', 't': 'Technology', 'm': 'Health'}

print("📊 Thống kê chi tiết các phân loại (đã giải mã):")

# Giả sử cột đó tên là 'category' hoặc 'type'
for col in ['category', 'type', 'subject']:
    if col in df_news.columns:
        # Tạo một bản sao để hiển thị cho đẹp
        display_series = df_news[col].map(mapping).fillna(df_news[col])
        print(f"\n--- Kết quả từ cột '{col}': ---")
        print(display_series.value_counts())

📊 Thống kê chi tiết các phân loại (đã giải mã):

--- Kết quả từ cột 'category': ---
category
Entertainment    187944
Business         144127
Technology       133706
Health            56642
Name: count, dtype: int64


In [28]:
# --- CHÚ THÍCH CÁC THÔNG SỐ ---
# nunique(): Đếm xem có bao nhiêu tác giả khác nhau (không tính trùng).
# value_counts(): Liệt kê tên tác giả và số lượng bài viết tương ứng.
# isnull().sum(): Kiểm tra xem có bao nhiêu bài báo không có tên tác giả.

print("✍️ ĐANG PHÂN TÍCH DỮ LIỆU TÁC GIẢ (AUTHOR)...")

if 'author' in df_news.columns:
    # 1. Tổng quan
    total_authors = df_news['author'].nunique()
    missing_authors = df_news['author'].isnull().sum()
    
    print(f"✅ Tổng số tác giả duy nhất: {total_authors}")
    print(f"⚠️ Số bài viết thiếu tên tác giả (NaN): {missing_authors}")
    print("-" * 30)
    
    # 2. Top 20 tác giả có nhiều bài viết nhất
    print("🔝 Top 20 tác giả đóng góp nhiều nhất:")
    print(df_news['author'].value_counts().head(20))
    
    # 3. Kiểm tra độ dài tên tác giả (để lọc rác nếu cần)
    # Đôi khi cột author chứa nguyên một đoạn văn hoặc link, mình check thử:
    print("-" * 30)
    print("📝 Ví dụ 5 tên tác giả đầu tiên trong danh sách:")
    print(df_news['author'].unique()[:5])

else:
    print("❌ Cảnh báo: Không tìm thấy cột 'author' trong DataFrame của ông!")
    print("Các cột hiện có là:", df_news.columns.tolist())

✍️ ĐANG PHÂN TÍCH DỮ LIỆU TÁC GIẢ (AUTHOR)...
❌ Cảnh báo: Không tìm thấy cột 'author' trong DataFrame của ông!
Các cột hiện có là: ['id', 'title', 'url', 'publisher', 'category', 'story', 'hostname', 'timestamp']
